# 02 - OFI Computation

Computing all three OFI variants and validating them.

1. Best-level OFI (vectorized, correct)
2. Multi-level OFI with depth normalization
3. PCA-integrated OFI
4. Temporal aggregation (1s, 5s, 30s)
5. OFI statistics and distribution analysis
6. OFI vs mid-price visualization

In [1]:
import importlib.util, os, pickle
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
os.makedirs('../data', exist_ok=True)

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, os.path.abspath(path))
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

ofi_mod = load_module('ofi', '../src/ofi.py')
viz_mod = load_module('viz', '../src/visualization.py')

df = ofi_mod.load_lob_data('../data/first_25000_rows.csv')
print(f'Loaded {len(df)} events')

Loaded 5000 events


## 1. Best-Level OFI

In [2]:
df['ofi_best'] = ofi_mod.compute_best_ofi(df)
print('Best-level OFI computed.')
print(df[['ts_event','mid_price','ofi_best']].head(10))
print(f'\nOFI stats: mean={df.ofi_best.mean():.4f}  std={df.ofi_best.std():.4f}')
print(f'Non-zero: {(df.ofi_best != 0).sum():,} / {len(df):,} events ({(df.ofi_best != 0).mean()*100:.1f}%)')

Best-level OFI computed.
                             ts_event  mid_price  ofi_best
0 2024-10-21 11:54:29.221064336+00:00    233.705         0
1 2024-10-21 11:54:29.223769812+00:00    233.705         2
2 2024-10-21 11:54:29.225030400+00:00    233.705         3
3 2024-10-21 11:54:29.712434212+00:00    233.705         0
4 2024-10-21 11:54:29.764673165+00:00    233.705         0
5 2024-10-21 11:54:29.764685027+00:00    233.705         0
6 2024-10-21 11:54:36.289427977+00:00    233.705         0
7 2024-10-21 11:54:37.990793976+00:00    233.700      -200
8 2024-10-21 11:54:39.124291588+00:00    233.710         1
9 2024-10-21 11:54:39.134140341+00:00    233.705      -200

OFI stats: mean=-14.8088  std=156.5319
Non-zero: 1,468 / 5,000 events (29.4%)


In [3]:
fig = viz_mod.ofi_price_chart(df, ofi_col='ofi_best', ticker='AAPL')
fig.show()

## 2. Multi-Level OFI

In [4]:
N_LEVELS = 9
ofi_levels_df = ofi_mod.compute_multilevel_ofi(df, max_levels=N_LEVELS, normalize=True)
level_cols = [c for c in ofi_levels_df.columns if c.startswith('ofi_level_')]
print(f'Multi-level OFI computed for {len(level_cols)} levels.')
print(ofi_levels_df[level_cols].describe().round(4))

Multi-level OFI computed for 9 levels.
       ofi_level_00  ofi_level_01  ofi_level_02  ofi_level_03  ofi_level_04  \
count     5000.0000     5000.0000     5000.0000     5000.0000     5000.0000   
mean        -0.0723       -0.0601       -0.0269        0.0865        0.0019   
std          0.7646        1.1889        1.3811        1.2898        1.1770   
min         -6.8724       -7.5939       -7.0139       -6.2802       -5.7475   
25%          0.0000        0.0000       -0.0044       -0.0039       -0.0041   
50%          0.0000        0.0000        0.0000        0.0000        0.0000   
75%          0.0000        0.0000        0.0436        0.0039        0.0041   
max          5.8614        6.6488        7.0139        7.0345        8.2394   

       ofi_level_05  ofi_level_06  ofi_level_07  ofi_level_08  
count     5000.0000     5000.0000     5000.0000     5000.0000  
mean         0.0786        0.1387        0.0203       -0.0197  
std          1.4463        1.2944        1.3914        1.

In [5]:
fig2 = viz_mod.multilevel_ofi_chart(ofi_levels_df, level_cols, df['ts_event'])
fig2.show()

## 3. PCA-Integrated OFI

In [6]:
ofi_int, weights, evr = ofi_mod.compute_integrated_ofi(ofi_levels_df, level_cols)
df['ofi_integrated'] = ofi_int.values

print(f'PCA Explained Variance Ratio: {evr*100:.1f}%')
print('This is the fraction of multi-level OFI variance captured by the first PC.')
print()
print('PCA Weights (L1 normalized):')
for col, w in zip(level_cols, weights):
    print(f'  {col}: {w:.4f}')

fig3 = viz_mod.pca_weights_chart(weights, level_cols, evr)
fig3.show()
print('\nInterpretation: Levels with larger |weight| contribute more to the integrated signal.')

PCA Explained Variance Ratio: 34.5%
This is the fraction of multi-level OFI variance captured by the first PC.

PCA Weights (L1 normalized):
  ofi_level_00: 0.0309
  ofi_level_01: 0.0771
  ofi_level_02: 0.1182
  ofi_level_03: 0.1342
  ofi_level_04: 0.1153
  ofi_level_05: 0.1654
  ofi_level_06: 0.1227
  ofi_level_07: 0.1191
  ofi_level_08: 0.1171



Interpretation: Levels with larger |weight| contribute more to the integrated signal.


## 4. Temporal Aggregation

In [7]:
# Add multi-level OFI to main df for aggregation
df_full = pd.concat([df.reset_index(drop=True),
                      ofi_levels_df.reset_index(drop=True)], axis=1)
df_full = df_full.loc[:, ~df_full.columns.duplicated()]

ofi_cols_all = ['ofi_best', 'ofi_integrated'] + level_cols
# Keep only cols that exist
ofi_cols_all = [c for c in ofi_cols_all if c in df_full.columns]

# Aggregate at different frequencies
agg_results = {}
for freq in ['1s', '5s', '30s']:
    agg = ofi_mod.aggregate_ofi(df_full, ofi_cols_all, freq=freq)
    agg_results[freq] = agg
    print(f'{freq} aggregation: {len(agg)} intervals  |  '
          f'avg OFI/interval: {agg["ofi_best"].abs().mean():.2f}')

# Save
import pickle
with open('../data/ofi_data.pkl', 'wb') as f:
    pickle.dump({
        'df':          df_full,
        'ofi_levels':  ofi_levels_df,
        'level_cols':  level_cols,
        'weights':     weights,
        'evr':         evr,
        'agg_results': agg_results,
    }, f)
print('\nSaved to data/ofi_data.pkl')

1s aggregation: 1273 intervals  |  avg OFI/interval: 110.79
5s aggregation: 643 intervals  |  avg OFI/interval: 188.03
30s aggregation: 141 intervals  |  avg OFI/interval: 678.71

Saved to data/ofi_data.pkl


In [8]:
# Visualize 1s aggregated OFI vs price
agg_1s = agg_results['1s']
fig4 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                      subplot_titles=['Mid Price (1s)', 'OFI (1s aggregate)'],
                      row_heights=[0.5, 0.5], vertical_spacing=0.05)

fig4.add_trace(go.Scatter(
    x=agg_1s['ts_event'], y=agg_1s['mid_price'],
    line=dict(color='#4C6EF5', width=1.5), name='Mid Price'
), row=1, col=1)

ofi_vals = agg_1s['ofi_best'].fillna(0)
fig4.add_trace(go.Bar(
    x=agg_1s['ts_event'], y=ofi_vals,
    marker_color=['#2F9E44' if v >= 0 else '#E03131' for v in ofi_vals],
    opacity=0.7, name='OFI (1s)'
), row=2, col=1)

fig4.update_layout(title='AAPL — 1-Second Aggregated OFI vs Mid Price',
                    height=500, template='plotly_white')
fig4.show()